# Pitcher D1 or Not Models — Train V2

Pitcher-specific D1 classification model. Pitchers have fundamentally different measurables
than hitters — velocity, spin rates, and secondary pitch arsenal replace bat/arm tools.

**Key differences from hitter models:**
- Primary signal is **fastball velocity** (Cohen's d ≈ 1.23 vs ~0.9 for hitter exit_velo)
- **Secondary pitch velocities** (changeup, curveball, slider) are strong predictors (~0.85 d)
- **Spin rates** add a quality dimension not present in hitter data
- **LHP premium**: 32% of P4 pitchers are lefties vs 24% of Non-D1 — handedness matters
- No batting metrics (exit_velo, distance, bat_speed) — purely pitching + physical

**Source:** `backend/ml/train/train_v2/pitchers/pitchers_cleaned.csv`

In [2]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

PITCHER_CSV = Path("pitchers_cleaned.csv")

pitchers = pd.read_csv(PITCHER_CSV, low_memory=False)
print(f"Loaded {len(pitchers):,} rows, {len(pitchers.columns)} columns")
print(f"\nPositions:")
print(pitchers["resolved_position"].value_counts())
print(f"\nCommitment groups:")
print(pitchers["committment_group"].value_counts())
print(f"\nClass distribution:")
print(pitchers["class"].value_counts().sort_index())
pitchers.head()

Loaded 19,506 rows, 44 columns

Positions:
resolved_position
RHP    14593
LHP     4913
Name: count, dtype: int64

Commitment groups:
committment_group
Non D1       11942
Non P4 D1     4772
P4            1469
Unknown       1323
Name: count, dtype: int64

Class distribution:
class
2022    3851
2023    3887
2024    3910
2025    3965
2026    3432
2027     461
Name: count, dtype: int64


,name,link,player_state,high_school,class,resolved_position,positions,commitment,commitment_date,height,...,thirty_time_date,ten_yard_time,ten_yard_time_date,run_speed_max,run_speed_max_date,player_region,conference,division,college_location,committment_group
0,Enmanuel Acevedo,https://www.prepbaseballreport.com/profiles/NY...,NY,Poly Prep,2027,RHP,RHP,Virginia,11/12/25,76.0,...,NaN,NaN,NaN,NaN,NaN,Northeast,Atlantic Coast Conference,NCAA I,"CHARLOTTESVILLE, VA",P4
1,Riley Ackerman,https://www.prepbaseballreport.com/profiles/IN...,IN,Crown Point,2027,LHP,LHP,Northwestern,10/21/25,75.0,...,NaN,NaN,NaN,NaN,NaN,Midwest,BIG 10,NCAA I,"EVANSTON, IL",Non P4 D1
2,Connor Ackerman,https://www.prepbaseballreport.com/profiles/NY...,NY,St. Dominic High School,2027,RHP,"RHP,1B",Hofstra,9/22/25,76.0,...,6/26/25,1.78,6/26/25,18.0,6/26/25,Northeast,Colonial,NCAA I,"HEMPSTEAD, NY",Non P4 D1
3,Jace Adams,https://www.prepbaseballreport.com/profiles/AZ...,AZ,Flagstaff,2027,RHP,"RHP,3B",Arizona State (8/03/25),NaN,75.0,...,NaN,NaN,NaN,NaN,NaN,West,Big 12,NCAA I,"TEMPE, AZ",P4
4,Noah Adkins,https://www.prepbaseballreport.com/profiles/FL...,FL,Hagerty,2027,RHP,RHP,Stetson,10/02/25,73.0,...,NaN,NaN,NaN,NaN,NaN,South,Atlantic Sun,NCAA I,"DELAND, FL",Non P4 D1


In [3]:
P_MODELING_COLS = [
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    "fastball_velo_max", "fastball_velo_max_date",
    "fastball_velo_range", "fastball_velo_range_date",
    "fastball_spin", "fastball_spin_date",
    "changeup_velo_range", "changeup_velo_range_date",
    "changeup_spin", "changeup_spin_date",
    "curveball_velo_range", "curveball_velo_range_date",
    "curveball_spin", "curveball_spin_date",
    "slider_velo_range", "slider_velo_range_date",
    "slider_spin", "slider_spin_date",
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "player_region", "committment_group", "commitment_date"
]

P_MODELING_COLS = [c for c in P_MODELING_COLS if c in pitchers.columns]
p_modeling = pitchers[P_MODELING_COLS]

p_modeling = p_modeling[p_modeling["committment_group"] != "Unknown"]

print(f"After removing Unknown:")
print(p_modeling["committment_group"].value_counts())
print(f"Shape: {p_modeling.shape}")

p_modeling["d1_or_not"] = (p_modeling["committment_group"] != "Non D1").astype(int)

print(f"\nTarget distribution:")
print(p_modeling["d1_or_not"].value_counts())
print(f"D1 rate: {p_modeling['d1_or_not'].mean():.1%}")

After removing Unknown:
committment_group
Non D1       11942
Non P4 D1     4772
P4            1469
Name: count, dtype: int64
Shape: (18183, 35)

Target distribution:
d1_or_not
0    11942
1     6241
Name: count, dtype: int64
D1 rate: 34.3%


## Data Exploration

Pitchers are measured very differently than hitters. The core tools are:
- **Fastball velocity** — the single most important metric for pitcher evaluation
- **Spin rates** — measure pitch quality and movement potential
- **Secondary pitch velocities** — changeup, curveball, slider measure arsenal depth
- **Physical measurements** — height/weight, with pitcher-specific importance (height = plane angle)

Key questions for exploration:
1. How much separation does each feature provide between D1 and Non-D1?
2. Which features are correlated (redundant)?
3. Does handedness (LHP vs RHP) affect the D1 signal?
4. How complete are spin rate measurements vs velocity measurements?

In [4]:
# ============================================================
# FEATURE DISTRIBUTIONS BY COMMITMENT GROUP
# ============================================================
EXPLORE_FEATURES = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
    "sixty_time",
]

print(f"{'='*90}")
print(f"FEATURE DISTRIBUTIONS BY COMMITMENT GROUP")
print(f"{'='*90}\n")

print(f"{'Feature':>25} {'Group':>12} {'N':>6} {'Mean':>8} {'Std':>7} {'Min':>7} {'P25':>7} {'P50':>7} {'P75':>7} {'Max':>7}")
print(f"{'-'*90}")

for feat in EXPLORE_FEATURES:
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = p_modeling["committment_group"] == group
        vals = p_modeling.loc[mask, feat].dropna()
        if len(vals) == 0:
            continue
        q = vals.quantile([0.25, 0.5, 0.75])
        print(f"{feat:>25} {group:>12} {len(vals):>6,} {vals.mean():>8.1f} {vals.std():>7.1f} "
              f"{vals.min():>7.1f} {q[0.25]:>7.1f} {q[0.5]:>7.1f} {q[0.75]:>7.1f} {vals.max():>7.1f}")
    print()

# Feature completeness
print(f"\n{'='*90}")
print(f"FEATURE COMPLETENESS (% non-null)")
print(f"{'='*90}\n")
print(f"{'Feature':>25} {'P4':>8} {'Non P4 D1':>12} {'Non D1':>10} {'Overall':>10}")
print(f"{'-'*70}")
for feat in EXPLORE_FEATURES:
    parts = []
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = p_modeling["committment_group"] == group
        n = mask.sum()
        has = p_modeling.loc[mask, feat].notna().sum()
        parts.append(f"{100*has/n:>7.0f}%")
    overall = 100 * p_modeling[feat].notna().sum() / len(p_modeling)
    parts.append(f"{overall:>9.0f}%")
    print(f"{feat:>25} {'  '.join(parts)}")

# Handedness breakdown
print(f"\n{'='*90}")
print(f"HANDEDNESS (LHP vs RHP) BY COMMITMENT GROUP")
print(f"{'='*90}\n")
ct = pd.crosstab(p_modeling["committment_group"], p_modeling["resolved_position"], normalize="index")
for group in ["P4", "Non P4 D1", "Non D1"]:
    lhp = ct.loc[group, "LHP"] * 100
    rhp = ct.loc[group, "RHP"] * 100
    n = (p_modeling["committment_group"] == group).sum()
    print(f"  {group:>12} (n={n:>5,}):  LHP={lhp:.1f}%  RHP={rhp:.1f}%")
print(f"\n  -> LHP premium at P4 level: {ct.loc['P4','LHP']*100 - ct.loc['Non D1','LHP']*100:+.1f}pp")

FEATURE DISTRIBUTIONS BY COMMITMENT GROUP

                  Feature        Group      N     Mean     Std     Min     P25     P50     P75     Max
------------------------------------------------------------------------------------------
                   height           P4  1,439     74.6     2.1    65.0    73.0    75.0    76.0    81.0
                   height    Non P4 D1  4,649     73.8     2.2    60.0    72.0    74.0    75.0    82.0
                   height       Non D1 11,758     72.9     2.4    58.0    71.0    73.0    75.0    83.0

                   weight           P4  1,316    194.5    19.5   125.0   180.0   195.0   208.0   265.0
                   weight    Non P4 D1  4,374    188.8    19.4   100.0   175.0   187.0   200.0   290.0
                   weight       Non D1 11,112    183.4    21.8   100.0   170.0   182.0   195.0   290.0

        fastball_velo_max           P4  1,287     91.1     3.5    75.0    89.0    91.0    93.0   102.0
        fastball_velo_max    Non P4 D1  

## Correlation Matrix

Understanding feature redundancy is critical for pitchers because:
- `fastball_velo_max` and `fastball_velo_range` are ~0.97 correlated (essentially the same measurement)
- Secondary pitch velocities correlate with fastball velocity (arm strength drives all)
- Spin rates may or may not track with velocity — the relationship varies by pitch type

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

CORR_FEATURES = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
    "sixty_time",
]

SHORT_NAMES = {
    "height": "height",
    "weight": "weight",
    "fastball_velo_max": "fb_velo_max",
    "fastball_velo_range": "fb_velo_rng",
    "fastball_spin": "fb_spin",
    "changeup_velo_range": "ch_velo",
    "changeup_spin": "ch_spin",
    "curveball_velo_range": "cb_velo",
    "curveball_spin": "cb_spin",
    "slider_velo_range": "sl_velo",
    "slider_spin": "sl_spin",
    "sixty_time": "sixty",
}

corr_data = p_modeling[CORR_FEATURES].rename(columns=SHORT_NAMES)
corr_matrix = corr_data.corr()

# ---- Visual heatmap ----
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Pitcher Feature Correlation Matrix", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("pitcher_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: pitcher_correlation_matrix.png\n")

# ---- Text version ----
print("CORRELATION MATRIX (Pearson r)")
print("=" * 90)
labels = list(corr_matrix.columns)
header = f"{'':>12}" + "".join(f"{l:>12}" for l in labels)
print(header)
print("-" * len(header))
for i, row_label in enumerate(labels):
    row_str = f"{row_label:>12}"
    for j, col_label in enumerate(labels):
        val = corr_matrix.iloc[i, j]
        if j > i:
            row_str += f"{'':>12}"
        else:
            marker = " **" if abs(val) > 0.7 and i != j else ""
            row_str += f"{val:>9.2f}{marker:>3}"
    print(row_str)

print(f"\n** = |r| > 0.70 (high collinearity)")

# ---- Key takeaways ----
print(f"\nKEY CORRELATIONS:")
print(f"  fb_velo_max ↔ fb_velo_range:  r = {corr_matrix.loc['fb_velo_max','fb_velo_rng']:.3f}  (near-duplicate — drop one)")
pairs = []
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            pairs.append((labels[i], labels[j], r))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for a, b, r in pairs[:10]:
    print(f"  {a:>12} ↔ {b:<12}:  r = {r:+.3f}")

Saved: pitcher_correlation_matrix.png

CORRELATION MATRIX (Pearson r)
                  height      weight fb_velo_max fb_velo_rng     fb_spin     ch_velo     ch_spin     cb_velo     cb_spin     sl_velo     sl_spin       sixty
------------------------------------------------------------------------------------------------------------------------------------------------------------
      height     1.00                                                                                                                                       
      weight     0.54        1.00                                                                                                                           
 fb_velo_max     0.30        0.32        1.00                                                                                                               
 fb_velo_rng     0.29        0.31        0.97 **     1.00                                                                                        

## Cohen's d — Effect Size Analysis

Cohen's d measures the standardized difference in means between D1 and Non-D1 pitchers.
This tells us how much "separation" each feature provides:
- **|d| > 0.8** = large effect (strong predictor)
- **0.5 < |d| < 0.8** = medium effect
- **0.2 < |d| < 0.5** = small effect
- **|d| < 0.2** = negligible

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COHENS_FEATURES = [
    "height", "weight",
    "fastball_velo_max", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
    "sixty_time",
]

# Compute Cohen's d for each feature (D1 vs Non-D1)
effects = []
for feat in COHENS_FEATURES:
    d1_vals = p_modeling.loc[p_modeling["d1_or_not"] == 1, feat].dropna()
    nd1_vals = p_modeling.loc[p_modeling["d1_or_not"] == 0, feat].dropna()
    pooled = np.sqrt((d1_vals.std()**2 + nd1_vals.std()**2) / 2)
    d = (d1_vals.mean() - nd1_vals.mean()) / pooled if pooled > 0 else 0
    effects.append({"feature": feat, "d": d, "abs_d": abs(d),
                     "d1_mean": d1_vals.mean(), "nd1_mean": nd1_vals.mean(),
                     "d1_n": len(d1_vals), "nd1_n": len(nd1_vals)})

effects_df = pd.DataFrame(effects).sort_values("abs_d", ascending=True)

# ---- Visual chart ----
fig, ax = plt.subplots(figsize=(10, 7))
colors = []
for d in effects_df["d"]:
    if abs(d) >= 0.8:
        colors.append("#2d8a4e" if d > 0 else "#c0392b")
    elif abs(d) >= 0.5:
        colors.append("#52a86b" if d > 0 else "#e74c3c")
    elif abs(d) >= 0.2:
        colors.append("#82c99a" if d > 0 else "#f1948a")
    else:
        colors.append("#bbb")

bars = ax.barh(range(len(effects_df)), effects_df["d"].values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(effects_df)))
ax.set_yticklabels(effects_df["feature"].values, fontsize=10)
ax.set_xlabel("Cohen's d (D1 vs Non-D1)", fontsize=11)
ax.set_title("Pitcher Feature Effect Sizes — D1 vs Non-D1", fontsize=13, fontweight="bold")

# Reference lines
for threshold, label in [(0.2, "small"), (0.5, "medium"), (0.8, "large")]:
    ax.axvline(threshold, color="#999", linestyle="--", linewidth=0.7, alpha=0.6)
    ax.axvline(-threshold, color="#999", linestyle="--", linewidth=0.7, alpha=0.6)
ax.axvline(0, color="black", linewidth=0.8)

# Value labels
for i, (_, row) in enumerate(effects_df.iterrows()):
    x_pos = row["d"]
    ha = "left" if x_pos >= 0 else "right"
    offset = 0.02 if x_pos >= 0 else -0.02
    ax.text(x_pos + offset, i, f"{row['d']:+.3f}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.set_xlim(-0.5, 1.5)
plt.tight_layout()
plt.savefig("pitcher_cohens_d.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: pitcher_cohens_d.png\n")

# ---- Text version ----
effects_sorted = sorted(effects, key=lambda x: x["abs_d"], reverse=True)

print("COHEN'S d — D1 vs Non-D1 PITCHERS")
print("=" * 90)
print(f"{'Feature':>25} {'Cohen d':>10} {'Effect':>10} {'D1 Mean':>10} {'Non-D1':>10} {'Diff':>8} {'D1 n':>7} {'ND1 n':>7}")
print(f"{'-'*90}")
for e in effects_sorted:
    if e["abs_d"] >= 0.8:
        size = "LARGE"
    elif e["abs_d"] >= 0.5:
        size = "medium"
    elif e["abs_d"] >= 0.2:
        size = "small"
    else:
        size = "negl."
    diff = e["d1_mean"] - e["nd1_mean"]
    print(f"{e['feature']:>25} {e['d']:>+10.3f} {size:>10} {e['d1_mean']:>10.1f} {e['nd1_mean']:>10.1f} "
          f"{diff:>+8.1f} {e['d1_n']:>7,} {e['nd1_n']:>7,}")

# ---- Interpretation ----
print(f"\nINTERPRETATION:")
large = [e for e in effects_sorted if e["abs_d"] >= 0.8]
medium = [e for e in effects_sorted if 0.5 <= e["abs_d"] < 0.8]
small = [e for e in effects_sorted if 0.2 <= e["abs_d"] < 0.5]
print(f"  Large effect (|d| >= 0.8):  {', '.join(e['feature'] for e in large)}")
print(f"  Medium effect:              {', '.join(e['feature'] for e in medium)}")
print(f"  Small effect:               {', '.join(e['feature'] for e in small)}")
print(f"\n  -> Fastball velocity dominates (d=1.23) — strongest single feature across ALL position models")
print(f"  -> Secondary pitch velocities cluster ~0.85 — arm strength drives everything")
print(f"  -> Spin rates provide medium signal — quality layer on top of velocity")
print(f"  -> Sixty time is weak for pitchers (d=-0.29) — not a primary tool for mound work")

# Also do P4 vs Non-P4 D1 comparison
print(f"\n\n{'='*90}")
print(f"COHEN'S d — P4 vs Non-P4 D1 PITCHERS (preview for P4 model)")
print(f"{'='*90}")
print(f"{'Feature':>25} {'Cohen d':>10} {'P4 Mean':>10} {'NP4D1 Mean':>12} {'Diff':>8}")
print(f"{'-'*70}")
for feat in COHENS_FEATURES:
    p4 = p_modeling.loc[p_modeling["committment_group"] == "P4", feat].dropna()
    np4 = p_modeling.loc[p_modeling["committment_group"] == "Non P4 D1", feat].dropna()
    pooled = np.sqrt((p4.std()**2 + np4.std()**2) / 2)
    d = (p4.mean() - np4.mean()) / pooled if pooled > 0 else 0
    diff = p4.mean() - np4.mean()
    print(f"{feat:>25} {d:>+10.3f} {p4.mean():>10.1f} {np4.mean():>12.1f} {diff:>+8.1f}")

Saved: pitcher_cohens_d.png

COHEN'S d — D1 vs Non-D1 PITCHERS
                  Feature    Cohen d     Effect    D1 Mean     Non-D1     Diff    D1 n   ND1 n
------------------------------------------------------------------------------------------
        fastball_velo_max     +1.233      LARGE       88.9       83.8     +5.1   5,248  10,177
      changeup_velo_range     +0.857      LARGE       78.6       74.8     +3.9   4,884   9,223
        slider_velo_range     +0.847      LARGE       76.1       72.4     +3.7   3,761   6,286
     curveball_velo_range     +0.827      LARGE       73.4       69.9     +3.5   4,979   9,287
            fastball_spin     +0.618     medium     2162.8     2043.5   +119.3   4,325   8,823
                   height     +0.466      small       74.0       72.9     +1.1   6,088  11,758
              slider_spin     +0.424      small     2233.1     2113.5   +119.7   3,032   5,608
           curveball_spin     +0.416      small     2184.2     2062.4   +121.8   3,475

In [7]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
STALE_MONTHS = 24
OUTLIER_STD_DEV = 2

p_model_recent = p_modeling.copy()
p_model_recent["class"] = pitchers.loc[p_model_recent.index, "class"]

STAT_DATE_PAIRS = [
    ("fastball_velo_max",    "fastball_velo_max_date"),
    ("fastball_velo_range",  "fastball_velo_range_date"),
    ("fastball_spin",        "fastball_spin_date"),
    ("changeup_velo_range",  "changeup_velo_range_date"),
    ("changeup_spin",        "changeup_spin_date"),
    ("curveball_velo_range", "curveball_velo_range_date"),
    ("curveball_spin",       "curveball_spin_date"),
    ("slider_velo_range",    "slider_velo_range_date"),
    ("slider_spin",          "slider_spin_date"),
    ("sixty_time",           "sixty_time_date"),
    ("thirty_time",          "thirty_time_date"),
    ("ten_yard_time",        "ten_yard_time_date"),
    ("run_speed_max",        "run_speed_max_date"),
]
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in p_model_recent.columns and d in p_model_recent.columns]

def parse_pbr_date(d):
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

p_model_recent["grad_date"] = pd.to_datetime(p_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    p_model_recent[parsed_col] = p_model_recent[date_col].apply(parse_pbr_date)
    p_model_recent[months_col] = ((p_model_recent["grad_date"] - p_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

print(f"{'='*70}")
print(f"STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = p_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val = p_model_recent.loc[mask, stat_col].notna()
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
print(f"\n{'='*70}")
print(f"OUTLIER SUMMARY  (+-{OUTLIER_STD_DEV} std dev from group mean)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = p_model_recent["committment_group"] == group
    tot_outliers = 0
    tot_stale_o = 0
    for stat_col in stat_value_cols:
        months_col = f"{stat_col}__months_before_grad"
        vals = p_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        tot_outliers += is_outlier.sum()
        tot_stale_o += (is_outlier & is_stale).sum()
    tot_fresh_o = tot_outliers - tot_stale_o
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers -> {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

internal_cols = [c for c in p_model_recent.columns if c.startswith("_")]
p_model_recent.drop(columns=internal_cols, inplace=True)
print(f"\np_model_recent shape: {p_model_recent.shape}")

Parsed 13 stat date columns.

STALENESS BY COMMITMENT GROUP  (threshold = 24 months)

            P4  (1,469 players): 3,002 / 10,804 values stale (27.8%)
     Non P4 D1  (4,772 players): 8,199 / 34,619 values stale (23.7%)
        Non D1  (11,942 players): 14,356 / 87,422 values stale (16.4%)

OUTLIER SUMMARY  (+-2 std dev from group mean)

            P4:  502 outliers -> 259 stale (52%), 243 fresh (48%)
     Non P4 D1: 1636 outliers -> 782 stale (48%), 854 fresh (52%)
        Non D1: 3880 outliers -> 1230 stale (32%), 2650 fresh (68%)

p_model_recent shape: (18183, 51)


In [8]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1", "Non D1"]:
            mask = p_model_recent["committment_group"] == group
            vals = p_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                p_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                p_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>25}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"\nTotal across {pass_i} passes: {grand_total} values NaN'd")

remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = p_model_recent["committment_group"] == group
        vals = p_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"p_model_recent shape: {p_model_recent.shape}")

Pass 1: NaN'd 2271 stale outlier values
            fastball_velo_max: 278
          fastball_velo_range: 270
                fastball_spin: 135
          changeup_velo_range: 336
                changeup_spin: 95
         curveball_velo_range: 323
               curveball_spin: 136
            slider_velo_range: 187
                  slider_spin: 64
                   sixty_time: 275
                  thirty_time: 60
                ten_yard_time: 55
                run_speed_max: 57
Pass 2: NaN'd 671 stale outlier values
            fastball_velo_max: 70
          fastball_velo_range: 85
                fastball_spin: 18
          changeup_velo_range: 112
                changeup_spin: 11
         curveball_velo_range: 97
               curveball_spin: 21
            slider_velo_range: 43
                  slider_spin: 13
                   sixty_time: 154
                  thirty_time: 19
                ten_yard_time: 10
                run_speed_max: 18
Pass 3: NaN'd 112 stale out

## Logistic Regression Baseline

Using 7 core pitcher features:
- **Physical:** height, weight
- **Primary arm:** fastball_velo_max, fastball_spin
- **Arsenal:** changeup_velo_range, curveball_velo_range, slider_velo_range

Dropping `fastball_velo_range` (r=0.97 with fastball_velo_max) and `sixty_time` (weak signal for pitchers, d=-0.29).
Spin rates for secondary pitches excluded for now — lower completeness and moderate signal.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
]

X = p_model_recent[FEATURES]
y = p_model_recent["d1_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Logistic Regression Baseline — 7 Features")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print("Feature coefficients (standardized):")
print(coefs)

Logistic Regression Baseline — 7 Features
              precision    recall  f1-score   support

      Non D1       0.83      0.71      0.77      2389
          D1       0.57      0.73      0.64      1248

    accuracy                           0.72      3637
   macro avg       0.70      0.72      0.70      3637
weighted avg       0.74      0.72      0.72      3637

[[1700  689]
 [ 343  905]] 

Feature coefficients (standardized):
weight                 -0.159300
changeup_velo_range    -0.062728
changeup_spin          -0.014295
slider_spin            -0.004621
curveball_spin          0.008637
slider_velo_range       0.085947
fastball_spin           0.088362
curveball_velo_range    0.156510
fastball_velo_range     0.169771
height                  0.281957
fastball_velo_max       1.194481
dtype: float64


## XGBoost — 7 Production Features + Calibration

Same 7 features as the logistic regression baseline. XGBoost handles NaN natively (no imputation needed)
and can capture non-linear relationships between pitch metrics.

In [10]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_XGB = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
]

X = p_model_recent[FEATURES_XGB]
y = p_model_recent["d1_or_not"]

neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non D1, {pos} D1  (ratio {neg/pos:.2f}:1)\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("XGBoost — 7 Features (holdout)")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")
print(f"  -> {auc:.0%} of the time, a random D1 player scores higher than a random Non-D1 player\n")

importances = pd.Series(xgb.feature_importances_, index=FEATURES_XGB).sort_values(ascending=False)
print("Feature importance:")
print(importances)

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    xgb, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\n\nStratified 5-Fold CV:")
print("=" * 55)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {train_scores.mean():.3f} (+/- {train_scores.std():.3f})  |  test {test_scores.mean():.3f} (+/- {test_scores.std():.3f})")

print(f"\n  Overfit gap (AUC): {cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean():.3f}")

# ============================================================
# Calibrated XGBoost — isotonic calibration
# ============================================================
xgb_calibrated = CalibratedClassifierCV(xgb, method="isotonic", cv=5)
xgb_calibrated.fit(X_train, y_train)

y_proba_cal = xgb_calibrated.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

print("\n\nCalibrated XGBoost — 7 Features (threshold=0.45)")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred_cal), "\n")

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=10)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)

print("Calibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)
print(f"\nBrier score:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

max_gap_raw = max(abs(mean_raw[i] - fraction_raw[i]) for i in range(len(mean_raw)))
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"Max calibration gap:  raw = {max_gap_raw:.2f}  |  calibrated = {max_gap_cal:.2f}")

auc_cal = roc_auc_score(y_test, y_proba_cal)
print(f"AUC-ROC (calibrated): {auc_cal:.4f}")

# Compare with LogReg baseline
xgb_report = classification_report(y_test, y_pred, target_names=["Non D1", "D1"], output_dict=True)
print(f"\n{'=' * 55}")
print("COMPARISON: LogReg baseline vs XGBoost + Calibration")
print(f"{'=' * 55}")
print(f"  XGBoost 7-feat:    D1 F1={xgb_report['D1']['f1-score']:.2f}  acc={xgb_report['accuracy']:.2f}  AUC={auc:.3f}  Brier(cal)={brier_cal:.4f}")

Class balance: 11942 Non D1, 6241 D1  (ratio 1.91:1)

XGBoost — 7 Features (holdout)
              precision    recall  f1-score   support

      Non D1       0.85      0.70      0.77      2389
          D1       0.57      0.76      0.65      1248

    accuracy                           0.72      3637
   macro avg       0.71      0.73      0.71      3637
weighted avg       0.75      0.72      0.73      3637

[[1675  714]
 [ 295  953]] 

AUC-ROC: 0.8127
  -> 81% of the time, a random D1 player scores higher than a random Non-D1 player

Feature importance:
fastball_velo_max       0.409570
fastball_velo_range     0.257919
curveball_velo_range    0.055235
height                  0.054940
changeup_velo_range     0.052021
fastball_spin           0.037962
weight                  0.031615
changeup_spin           0.030731
slider_spin             0.025640
slider_velo_range       0.023950
curveball_spin          0.020416
dtype: float32


Stratified 5-Fold CV:
       AUC-ROC:  train 0.842 (+/- 0.0

## Feature Engineering — Pitcher Interactions

Pitcher-specific interaction features that capture scouting concepts:
- **Velocity differential** (fastball - offspeed) — tunneling/deception ability
- **Spin efficiency proxy** (spin / velo) — how much "stuff" per mph
- **Arsenal spread** — range between fastest and slowest pitch
- **Power + spin combo** — velocity × spin rate
- **BMI proxy** — body composition

In [11]:
CORE_7 = ["height", "weight", "fastball_velo_max", "fastball_spin",
          "changeup_velo_range", "curveball_velo_range", "slider_velo_range"]

p_eng = p_model_recent[CORE_7 + ["d1_or_not"]].copy()

# --- Interaction features ---

# Velocity differential: fastball - changeup (tunneling/deception)
p_eng["fb_ch_diff"] = p_eng["fastball_velo_max"] - p_eng["changeup_velo_range"]

# Velocity differential: fastball - curveball (vertical plane separation)
p_eng["fb_cb_diff"] = p_eng["fastball_velo_max"] - p_eng["curveball_velo_range"]

# Spin efficiency proxy: how much spin per mph of fastball
p_eng["spin_per_mph"] = p_eng["fastball_spin"] / p_eng["fastball_velo_max"]

# Raw power × spin: velocity × spin = overall fastball quality
p_eng["velo_x_spin"] = p_eng["fastball_velo_max"] * p_eng["fastball_spin"]

# Arsenal spread: range between fastest pitch and slowest breaking ball
p_eng["arsenal_spread"] = p_eng["fastball_velo_max"] - p_eng["curveball_velo_range"]

# BMI proxy
p_eng["bmi_proxy"] = p_eng["weight"] / (p_eng["height"] ** 2) * 703

# --- Check separation ---
ENG_FEATURES = [c for c in p_eng.columns if c not in ["d1_or_not"]]

print(f"p_eng: {p_eng.shape[0]} rows, {len(ENG_FEATURES)} features")
print(f"  Core: {len(CORE_7)}  |  Engineered: {len(ENG_FEATURES) - len(CORE_7)}\n")

print("Cohen's d (sorted by |d|):")
print("=" * 55)
effects = []
for feat in ENG_FEATURES:
    d1 = p_eng.loc[p_eng["d1_or_not"] == 1, feat].dropna()
    nd1 = p_eng.loc[p_eng["d1_or_not"] == 0, feat].dropna()
    pooled = np.sqrt((nd1.std()**2 + d1.std()**2) / 2)
    d = (d1.mean() - nd1.mean()) / pooled if pooled > 0 else 0
    effects.append((feat, d))
for feat, d in sorted(effects, key=lambda x: abs(x[1]), reverse=True):
    tag = " *" if feat not in CORE_7 else ""
    print(f"  {feat:>22}  d = {d:+.3f}{tag}")

print("\n* = engineered feature")

print("\nEngineered feature correlations with parents:")
print("=" * 55)
eng_pairs = [
    ("fb_ch_diff",      ["fastball_velo_max", "changeup_velo_range"]),
    ("fb_cb_diff",      ["fastball_velo_max", "curveball_velo_range"]),
    ("spin_per_mph",    ["fastball_spin", "fastball_velo_max"]),
    ("velo_x_spin",     ["fastball_velo_max", "fastball_spin"]),
    ("arsenal_spread",  ["fastball_velo_max", "curveball_velo_range"]),
    ("bmi_proxy",       ["height", "weight"]),
]
for eng, parents in eng_pairs:
    corrs = [p_eng[eng].corr(p_eng[p]) for p in parents]
    corr_str = ", ".join(f"{p}={c:.2f}" for p, c in zip(parents, corrs))
    print(f"  {eng:>18} -> {corr_str}")

p_eng: 18183 rows, 13 features
  Core: 7  |  Engineered: 6

Cohen's d (sorted by |d|):
       fastball_velo_max  d = +1.356
             velo_x_spin  d = +1.022 *
     changeup_velo_range  d = +0.985
    curveball_velo_range  d = +0.947
       slider_velo_range  d = +0.924
           fastball_spin  d = +0.658
                  height  d = +0.466
              fb_cb_diff  d = +0.450 *
          arsenal_spread  d = +0.450 *
              fb_ch_diff  d = +0.381 *
                  weight  d = +0.323
               bmi_proxy  d = +0.065 *
            spin_per_mph  d = +0.011 *

* = engineered feature

Engineered feature correlations with parents:
          fb_ch_diff -> fastball_velo_max=0.34, changeup_velo_range=-0.31
          fb_cb_diff -> fastball_velo_max=0.45, curveball_velo_range=-0.28
        spin_per_mph -> fastball_spin=0.83, fastball_velo_max=-0.04
         velo_x_spin -> fastball_velo_max=0.78, fastball_spin=0.94
      arsenal_spread -> fastball_velo_max=0.45, curveball_velo_ra

In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# ============================================================
# XGBoost — Engineered Features + Calibration
# ============================================================
FEATURES_ENG = [c for c in p_eng.columns if c != "d1_or_not"]

X = p_eng[FEATURES_ENG]
y = p_eng["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_eng = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb_eng.fit(X_train, y_train)
y_pred = xgb_eng.predict(X_test)
y_proba = xgb_eng.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_ENG)} Features (7 core + {len(FEATURES_ENG)-7} engineered)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred))

auc_eng = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc_eng:.4f}")

importances = pd.Series(xgb_eng.feature_importances_, index=FEATURES_ENG).sort_values(ascending=False)
print(f"\nFeature importance:")
print(importances.to_string())

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_eng, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\nStratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"\n  Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# Calibration
xgb_eng_cal = CalibratedClassifierCV(xgb_eng, method="isotonic", cv=5)
xgb_eng_cal.fit(X_train, y_train)
y_proba_cal = xgb_eng_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)
brier_raw_eng = brier_score_loss(y_test, y_proba)
brier_cal_eng = brier_score_loss(y_test, y_proba_cal)

print(f"\nCalibrated model (threshold=0.45):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(f"Brier:  raw = {brier_raw_eng:.4f}  |  calibrated = {brier_cal_eng:.4f}")
print(f"\nCalibration curve (calibrated):")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

auc_eng_cal = roc_auc_score(y_test, y_proba_cal)
print(f"\nAUC-ROC (calibrated): {auc_eng_cal:.4f}")

print(f"\n{'=' * 60}")
print(f"COMPARISON: 7-feature base vs {len(FEATURES_ENG)}-feature engineered")
print(f"{'=' * 60}")
print(f"  7-feat base:      AUC={auc:.4f}  Brier(cal)={brier_cal:.4f}")
print(f"  {len(FEATURES_ENG)}-feat eng:    AUC={auc_eng:.4f}  Brier(cal)={brier_cal_eng:.4f}")
print(f"  AUC delta:        {auc_eng - auc:+.4f}")

XGBoost — 13 Features (7 core + 6 engineered)
              precision    recall  f1-score   support

      Non D1       0.85      0.70      0.77      2389
          D1       0.57      0.76      0.65      1248

    accuracy                           0.72      3637
   macro avg       0.71      0.73      0.71      3637
weighted avg       0.75      0.72      0.73      3637

[[1672  717]
 [ 302  946]]

AUC-ROC: 0.8100

Feature importance:
fastball_velo_max       0.412465
velo_x_spin             0.134377
curveball_velo_range    0.081821
height                  0.056506
changeup_velo_range     0.055275
fb_cb_diff              0.052262
arsenal_spread          0.041247
fb_ch_diff              0.033848
bmi_proxy               0.030880
weight                  0.029960
spin_per_mph            0.024563
slider_velo_range       0.023801
fastball_spin           0.022993

Stratified 5-Fold CV:
       AUC-ROC:  train 0.840 (+/- 0.002)  |  test 0.825 (+/- 0.006)
      accuracy:  train 0.737 (+/- 0.003)  

In [13]:
import joblib
import json
import os
from datetime import datetime

# ============================================================
# Save calibrated XGBoost D1 model to production model directory
# ============================================================
THRESHOLD = 0.45
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_p", "models_d1_or_not_p", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

FEATURES_SAVE = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
]

X = p_model_recent[FEATURES_SAVE]
y = p_model_recent["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_final = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)
xgb_final.fit(X_train, y_train)

xgb_final_cal = CalibratedClassifierCV(xgb_final, method="isotonic", cv=5)
xgb_final_cal.fit(X_train, y_train)

# --- Eval metrics on holdout ---
y_proba_test = xgb_final_cal.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= THRESHOLD).astype(int)

auc = roc_auc_score(y_test, y_proba_test)
brier = brier_score_loss(y_test, y_proba_test)
report = classification_report(y_test, y_pred_test, target_names=["Non D1", "D1"], output_dict=True)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_test, n_bins=10)
max_cal_gap = max(abs(m - f) for m, f in zip(mean_cal, fraction_cal))

# --- Save model ---
joblib.dump(xgb_final_cal, os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl"))
print(f"Saved: calibrated_xgb_model.pkl")

# --- Save model config ---
model_config = {
    "model_version": f"p_d1_xgb_cal_{VERSION}",
    "creation_date": datetime.now().isoformat(),
    "model_type": "calibrated_xgboost_pitcher_d1",
    "calibration_method": "isotonic",
    "threshold": THRESHOLD,
    "features": FEATURES_SAVE,
    "hyperparameters": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.01,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 3.0,
        "min_child_weight": 5,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "performance_metrics": {
        "auc_roc": round(auc, 4),
        "brier_score": round(brier, 4),
        "max_calibration_gap": round(max_cal_gap, 4),
        "d1_precision": round(report["D1"]["precision"], 4),
        "d1_recall": round(report["D1"]["recall"], 4),
        "d1_f1": round(report["D1"]["f1-score"], 4),
        "accuracy": round(report["accuracy"], 4),
        "threshold_used": THRESHOLD,
    },
    "dataset_info": {
        "total_samples": len(p_model_recent),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "d1_rate": round(pos / (neg + pos), 4),
        "stale_cleaning": "Option B multi-pass (stale outliers only, +/-2 std + >24mo)",
    },
    "calibration_curve": {
        "predicted": [round(float(m), 4) for m in mean_cal],
        "actual": [round(float(f), 4) for f in fraction_cal],
    },
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: model_config.json")

# --- Save feature metadata ---
feature_metadata = {
    "features": FEATURES_SAVE,
    "required_columns": FEATURES_SAVE,
    "notes": "XGBoost handles NaN natively. No imputation or scaling needed. 11 pitcher features: physical (2) + fastball velo/spin (3) + secondary pitch velo/spin (6). Players missing secondary pitches or spin data still get valid predictions.",
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved: feature_metadata.json")

# --- Print summary ---
print(f"\nModel saved to: {SAVE_DIR}")
print(f"\n{'=' * 60}")
print(f"PRODUCTION MODEL METRICS (threshold={THRESHOLD})")
print(f"{'=' * 60}")
print(f"  AUC-ROC:           {auc:.4f}")
print(f"  Brier score:       {brier:.4f}")
print(f"  Max cal gap:       {max_cal_gap:.4f}")
print(f"  D1 Precision:      {report['D1']['precision']:.4f}")
print(f"  D1 Recall:         {report['D1']['recall']:.4f}")
print(f"  D1 F1:             {report['D1']['f1-score']:.4f}")
print(f"  Accuracy:          {report['accuracy']:.4f}")
print(f"  Features:          {len(FEATURES_SAVE)} ({', '.join(FEATURES_SAVE)})")
print(f"  Threshold:         {THRESHOLD}")
print(f"  Training samples:  {len(X_train)}")
print(f"  Test samples:      {len(X_test)}")
print(f"  D1 base rate:      {pos/(neg+pos):.1%}")

print(f"\nCalibration curve:")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

print(f"\nFiles in {VERSION}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<35} {size:>8,} bytes")

Saved: calibrated_xgb_model.pkl
Saved: model_config.json
Saved: feature_metadata.json

Model saved to: /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_p/models_d1_or_not_p/version_04212026

PRODUCTION MODEL METRICS (threshold=0.45)
  AUC-ROC:           0.8125
  Brier score:       0.1645
  Max cal gap:       0.0446
  D1 Precision:      0.6226
  D1 Recall:         0.6082
  D1 F1:             0.6153
  Accuracy:          0.7391
  Features:          11 (height, weight, fastball_velo_max, fastball_velo_range, fastball_spin, changeup_velo_range, changeup_spin, curveball_velo_range, curveball_spin, slider_velo_range, slider_spin)
  Threshold:         0.45
  Training samples:  14546
  Test samples:      3637
  D1 base rate:      34.3%

Calibration curve:
   Predicted    Actual       Gap
        0.04      0.05     -0.01
        0.14      0.17     -0.03
        0.25      0.29     -0.04
        0.35      0.39     -0.04
        0.45      0.44     +0.01
     